# Day 80: Dr Semmelweis

Solution notebook. One section per lecture.

## Lecture 592: Preliminary Data Exploration and Visualising Births & Deaths

In [ ]:
import matplotlib.pyplot as plt

from semmelweis.domain import exploration
from semmelweis.infrastructure.csv_loader import load_monthly, load_yearly
from semmelweis.presentation.charts import births_and_deaths_chart

df_yearly = load_yearly()
df_monthly = load_monthly()

### Preliminary Data Exploration

In [ ]:
print(exploration.table_shape(df_yearly), exploration.table_shape(df_monthly))
print(exploration.column_names(df_yearly))
print(exploration.column_names(df_monthly))
print(exploration.years_covered(df_yearly))
print(df_monthly['date'].min(), df_monthly['date'].max())

In [ ]:
print('NaN yearly:', exploration.has_nan_values(df_yearly))
print('NaN monthly:', exploration.has_nan_values(df_monthly))
print('Duplicates yearly:', exploration.has_duplicates(df_yearly))
print('Duplicates monthly:', exploration.has_duplicates(df_monthly))

In [ ]:
print(f'Average births per month: {exploration.average_births_per_month(df_monthly):.2f}')
print(f'Average deaths per month: {exploration.average_deaths_per_month(df_monthly):.2f}')

### Percentage of Women Dying in Childbirth

In [ ]:
print(f'{exploration.percentage_of_women_dying(df_yearly):.2f}%')

### Visualising Births and Deaths with Twin Axes

In [ ]:
births_and_deaths_chart(df_monthly)
plt.show()

## Lecture 593: The Yearly Data Split by Clinic

In [ ]:
from semmelweis.domain import clinic_split
from semmelweis.presentation.charts import (
    births_by_clinic_chart,
    deaths_by_clinic_chart,
    pct_deaths_by_clinic_chart,
)

### Births by Clinic

Which clinic is bigger? Did the hospital get more patients over time?

In [ ]:
births_by_clinic_chart(df_yearly)
plt.show()

In [ ]:
print(clinic_split.total_births_by_clinic(df_yearly))
print(clinic_split.busiest_clinic(df_yearly))

### Deaths by Clinic

What was the highest number of deaths in clinic 1 and in clinic 2?

In [ ]:
deaths_by_clinic_chart(df_yearly)
plt.show()

In [ ]:
print(clinic_split.highest_deaths_by_clinic(df_yearly))

### Proportion of Deaths and the Average Death Rate

In [ ]:
df_yearly = clinic_split.add_pct_deaths(df_yearly)
df_yearly.head()

In [ ]:
for clinic in ("clinic 1", "clinic 2"):
    rate = clinic_split.average_death_rate(df_yearly, clinic)
    print(f'Average death rate in {clinic}: {rate:.2f}%')

In [ ]:
pct_deaths_by_clinic_chart(df_yearly)
plt.show()

In [ ]:
print(clinic_split.clinic_with_higher_death_rate(df_yearly))

## Lecture 594: The Effect of Handwashing

In [ ]:
from semmelweis.domain import handwashing
from semmelweis.presentation.charts import handwashing_chart

### Monthly Proportion of Deaths

In [ ]:
df_monthly = handwashing.add_pct_deaths_monthly(df_monthly)
df_monthly.head()

### Before and After Handwashing

Handwashing became mandatory on 1847-06-01.

In [ ]:
before, after = handwashing.split_before_after(df_monthly)
print(len(before), len(after))

In [ ]:
rate_before, rate_after = handwashing.average_rate_before_after(df_monthly)
print(f'Average death rate before: {rate_before:.2f}%')
print(f'Average death rate after: {rate_after:.2f}%')
mean_before, mean_after = handwashing.mean_of_rates_before_after(df_monthly)
print(f'Mean of monthly rates before: {mean_before:.2f}%')
print(f'Mean of monthly rates after: {mean_after:.2f}%')
print(f'Handwashing cut the rate {handwashing.improvement_factor(rate_before, rate_after):.1f} times')

### The Effect on the Death Rate

In [ ]:
handwashing_chart(df_monthly)
plt.show()

## Lecture 595: Visualising Distributions and Testing for Statistical Significance

In [ ]:
from semmelweis.domain import significance
from semmelweis.presentation.charts import (
    box_plot_chart,
    histogram_overlay_chart,
    kde_chart,
)

df_pct = handwashing.add_pct_deaths_monthly(df_monthly)
df_washing = significance.add_washing_hands(df_pct)
df_washing.head()

### Mean Death Rate With and Without Handwashing

In [ ]:
without, with_washing = significance.group_means(df_washing)
print(f'Mean without handwashing: {without:.2%}')
print(f'Mean with handwashing: {with_washing:.2%}')

### Box Plot

In [ ]:
box_plot_chart(df_monthly)
plt.show()

### Superimposed Histograms

Both periods use 30 bins with the same edges; each bar is the percent of that period's months.

In [ ]:
histogram_overlay_chart(df_monthly)
plt.show()

### Kernel Density Estimate

The KDE estimates the probability density behind the sample. It is limited to the range of the data, so no density is drawn below the minimum or above the maximum.

In [ ]:
kde_chart(df_monthly)
plt.show()

### Statistical Significance

The t-test asks whether the two means could differ by chance. A p-value below 0.01 means the difference is significant at the 99% level.

In [ ]:
rates = df_washing['pct_deaths']
is_washing = df_washing['washing_hands']
result = significance.two_sample_t_test(rates[~is_washing], rates[is_washing])
print(f't-statistic: {result.statistic:.3f}')
print(f'p-value: {result.p_value:.3e}')
print('Significant at the 99% level:', significance.is_significant(result.p_value))